In [ ]:
%cd ../../
%load_ext dotenv
%dotenv

/Users/hoangle/Projects/untangling-people/ylva/fwo_models


In [2]:
from pathlib import Path
from datetime import datetime

import pandas as pd
import polars as pl
import numpy as np
from darts import TimeSeries
from darts.models import NaiveMovingAverage

/Users/hoangle/Projects/untangling-people/ylva/fwo_models/.venv/lib/python3.11/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


# Read data

In [5]:
restaurant_id = 3

In [6]:
PATH_ORDER_HISTORICAL = Path("data/processed/pos/historical/*.xlsx")

schema = {
    'date': pl.Date,
    'meal_id': pl.Int64,
    'restaurant_id': pl.Int64,
    'pcs': pl.Int64,
    'src': pl.String,
}

order_raw = pl.read_excel(PATH_ORDER_HISTORICAL, schema_overrides=schema)

order_raw.head()

date,meal_id,restaurant_id,pcs,src
date,i64,i64,i64,str
2025-09-01,141,4,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,276,4,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,214,4,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,39,4,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,266,4,2,"""Data Viikuna 9-2025.csv"""


# Forecast

In [5]:
cols_tgt = ['pcs']
cols_cov_date = [
    'weekday',
    'day',
    'month',
    'year',
    'weekday_sin',
    'weekday_cos',
    'day_sin',
    'day_cos',
    'month_sin',
    'month_cos',
]

orders = (
    order_raw

    .sort('date')

    .with_columns(
        *[pl.col(col).fill_null(0.) for col in cols_tgt]
    )

    .select(
        'restaurant_id',
        'meal_id',
        *[pl.struct('date', col).alias(col) for col in cols_tgt]
    )
    .group_by('restaurant_id', 'meal_id')
    .agg(
        *[pl.concat_list(col).flatten() for col in cols_tgt]
    )
)

orders.head()

restaurant_id,meal_id,pcs
i64,i64,list[struct[2]]
4,229,"[{2024-09-09,164.0}, {2024-09-10,31.0}, … {2025-03-27,67.0}]"
2,98,"[{2023-02-28,92.0}, {2023-03-01,17.0}, … {2024-04-18,21.0}]"
1,47,"[{2023-01-13,188.0}, {2023-01-24,168.0}, … {2025-09-12,70.0}]"
4,81,"[{2023-04-05,308.0}, {2023-05-17,98.0}, … {2025-03-12,13.0}]"
3,316,"[{2023-01-09,8.0}, {2023-01-10,11.0}, … {2025-09-30,18.0}]"


In [6]:
series_pair = {}

for entry in orders.iter_rows(named=True):

    # if entry['dish_id'] == 1057 and entry['restaurant_id'] == 0:
    #     break

    df: pd.DataFrame|None = None
    for idx, col in enumerate(cols_tgt):
        df_ = pd.DataFrame.from_records(entry[col]).sort_values('date')
        if df is None:
            df = df_
        else:
            df = df.merge(df_, on='date', how='inner')

    assert df is not None

    df['restaurant_id'] = entry['restaurant_id']
    df['date'] = pd.to_datetime(df['date'])
    df['weekday'] = df['date'].dt.day
    df['day'] = df['date'].dt.weekday
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['weekday_sin'] = np.sin(df['date'].dt.weekday * 2 * np.pi / 7)
    df['weekday_cos'] = np.cos(df['date'].dt.weekday * 2 * np.pi / 7)
    df['day_sin'] = np.sin(df['date'].dt.day * 2 * np.pi / 31)
    df['day_cos'] = np.cos(df['date'].dt.day * 2 * np.pi / 31)
    df['month_sin'] = np.sin(df['date'].dt.month * 2 * np.pi / 12)
    df['month_cos'] = np.cos(df['date'].dt.month * 2 * np.pi / 12)

    if len(df) == 0:
        continue

    series_tgt = TimeSeries.from_dataframe(
        df,
        value_cols=cols_tgt,
        fillna_value=False,
        static_covariates=df['restaurant_id']
    )
    series_cov_date = TimeSeries.from_dataframe(
        df,
        value_cols=cols_cov_date,
        fillna_value=False,
        static_covariates=df['restaurant_id']
    )

    series_pair[(entry['meal_id'], entry['restaurant_id'])] = {
        'dates': df['date'],
        'tgt': series_tgt,
        'cov': series_cov_date,
        'n_forecast': 1
    }


In [9]:
dish_id_forecast = []
for meal_id, entry in series_pair.items():
    if entry['n_forecast'] > 0:
        # print("Found")
        
        dish_id_forecast.append(meal_id)



forecast_list = []
for meal_id in dish_id_forecast:
    entry = series_pair[meal_id]

    series_tgt = entry['tgt']
    n_forecast = entry['n_forecast']
    # series_cov_date = entry['cov']
    # series_dates = entry['dates']

    model = NaiveMovingAverage(min(5, len(series_tgt)))
    model.fit(series_tgt)
    preds = model.predict(n_forecast)

    df_preds = preds.to_dataframe()
    df_preds['restaurant_id'] = int(series_tgt.static_covariates_values()[0][0])
    df_preds['meal_id'] = meal_id[0]

    forecast_list.append(df_preds)

forecast = (
    pl.from_dataframe(pd.concat(forecast_list))

    # Reformat columns
    .select(
        'meal_id', 'restaurant_id',
        pl.col('pcs').cast(pl.Int64),
    )
)
forecast.head()

meal_id,restaurant_id,pcs
i64,i64,i64
229,4,65
98,2,74
47,1,36
81,4,21
316,3,7


Save

In [12]:
PATH_DIR_FORECASTED = Path("data/processed/pos/forecasted")

In [13]:
cols = [
    'meal_id', 'restaurant_id', 'pcs',
]

In [14]:
path_forecasted = PATH_DIR_FORECASTED / f"{datetime.now().strftime(r'%m%d')}.xlsx"
path_forecasted.parent.mkdir(exist_ok=True, parents=True)

forecast.select(cols).write_excel(path_forecasted)